In [1]:
# from numba import cuda
import paicos as pa
import numpy as np
import cupy as cp
import turbocluster as tc
import math
from numba import cuda
import nvtx
import finufft

import cmasher as cmr
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize

pa.settings.strict_units = False



In [2]:
# A snapshot object
# snap = pa.Snapshot(pa.data_dir, 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/adiabatic-mhd/zoom4_ics_v1/output', 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
# snap = pa.Snapshot('/scratch/lperrone/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
snap = pa.Snapshot('/llust21/cosmo-plasm/zoom-simulations-arepo2/halo_0003/tng/zoom8/output', 
                   305, basename='snapshot')
center = snap.Cat.Group['GroupPos'][0]
widths = np.array([1e1, 8e0, 2e1], dtype=float)


Gauss = snap.uq('G')
arepo_length = snap.length
density_unit = snap['0_Density'].uq
L_box = (arepo_length*(widths[0]))

In [3]:
pe = tc.PotentialEnergy(snap, center, widths, snap['0_Coordinates'], snap['0_Masses'], npix=32)

/lustre/lperrone/miniforge3/envs/paicos-conda/lib/python3.11/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/lustre/lperrone/miniforge3/envs/paicos-conda/lib/python3.11/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/lustre/lperrone/miniforge3/envs/paicos-conda/lib/python3.11/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [4]:
np.sum(pe.tile.mass_per_tile)

array(0.19303647)

In [5]:
pe.tile.mass_per_tile.shape

(32, 25, 64)

In [6]:
np.sum(snap['0_Masses'][pe.index])

<PaicosQuantity 0.19303647 arepo_mass / small_h>

In [7]:
np.sum(pe.index)

62

In [8]:
np.sum(pe.tile.particles_per_tile)

array(62)

In [9]:
np.histogram(pe.tile.particles_per_tile.flatten())

(array([51138,     0,     0,     0,     0,     0,     0,     0,     0,
           62]),
 array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ]))

In [10]:
smoothing_length = 0.05*cp.asnumpy(pe.tile.tile_widths).min()*arepo_length
# smoothing_length = np.cbrt((snap["0_Volume"]) / (4.0 * np.pi / 3.0))

In [11]:
smoothing_length

<PaicosQuantity 0.015625 arepo_length small_a / small_h>

In [12]:
np.nonzero(pe.tile.mass_per_tile)

(array([ 0,  0,  0,  0,  0,  1,  2,  2,  3,  3,  3,  3,  4,  6,  6,  6,  8,
         9,  9,  9, 10, 10, 10, 11, 12, 12, 12, 12, 12, 14, 14, 14, 15, 15,
        15, 16, 17, 17, 17, 18, 18, 19, 19, 20, 20, 20, 20, 21, 22, 22, 23,
        24, 25, 25, 25, 26, 26, 26, 27, 27, 30, 31]),
 array([ 2,  9, 14, 16, 18,  7,  1,  3, 12, 14, 21, 23,  3,  5, 16, 23,  2,
         4, 10, 16,  2, 17, 23,  4,  2,  9, 13, 13, 17, 11, 18, 19,  0, 22,
        23, 11,  3,  3, 20,  4, 17,  1, 13,  7, 12, 20, 21, 15,  3, 24,  3,
        21,  3, 12, 12,  5,  8, 14, 10, 19, 19, 10]),
 array([25, 60, 47, 58, 38, 45, 31, 53, 36,  6, 33, 47,  1, 18, 14, 21, 61,
        23, 28,  2,  7, 30, 27, 38, 52, 12, 36, 47, 60, 26, 36, 53, 63,  6,
        44, 42,  1, 26, 19, 19, 49, 54, 34, 33, 56, 12, 27,  7, 10, 35, 45,
        61,  2, 13, 25, 28, 49, 42, 59, 33, 46, 40]))

In [13]:
print("%.12f"%(pe.tile.mass_per_tile[0,9,60]))

0.003154683392


In [14]:
pe.tile.start_index_for_tile[0,9,60]

array(57)

In [15]:
pe.tile.unsort_index[57]

array(8)

In [16]:
pe.mass.shape

(62,)

In [17]:
pe.mass[18], pe.pos[18]

(<PaicosQuantity 0.00344667 arepo_mass / small_h>,
 <PaicosQuantity [499820.51077706, 499882.13790958, 499709.95333704] arepo_length small_a / small_h>)

In [18]:
snap['0_Masses'][18], snap['0_Coordinates'][18]

(<PaicosQuantity 0.00310454 arepo_mass / small_h>,
 <PaicosQuantity [499804.810128  , 499900.03950425, 499742.54717684] arepo_length small_a / small_h>)

In [19]:
snap['0_Masses'][57], snap['0_Coordinates'][57]

(<PaicosQuantity 0.00373047 arepo_mass / small_h>,
 <PaicosQuantity [499830.73099683, 499888.76321796, 499731.36684294] arepo_length small_a / small_h>)

In [20]:
pe.index[57]

False

In [21]:
pe.tile.tile_index.shape

(62, 3)

In [29]:
tile_index_orig = pe.tile.tile_index[pe.tile.unsort_index]

In [54]:
for nx, ny, nz in tile_index_orig:
    # pe.tile.particles_per_tile[nx, ny, nz]
    start_idx = pe.tile.start_index_for_tile[nx, ny, nz]
    print('mass per tile: ', pe.tile.mass_per_tile[nx, ny, nz])
    ip_orig = int(pe.tile.sort_index[start_idx])
    # print('%s'%(pe.index[ip_orig]), snap['0_Masses'][ip_orig], snap['0_Coordinates'][ip_orig])
    print(snap['0_Masses'][pe.index][ip_orig], snap['0_Coordinates'][pe.index][ip_orig])
    

mass per tile:  0.0031196146737784147
0.0031196146737784147 arepo_mass / small_h [499826.88764569 499883.64783809 499712.60758361] arepo_length small_a / small_h
mass per tile:  0.0024896685499697924
0.0024896685499697924 arepo_mass / small_h [499826.49567412 499885.69587623 499712.80440248] arepo_length small_a / small_h
mass per tile:  0.0030036591924726963
0.0030036591924726963 arepo_mass / small_h [499824.90004678 499885.1062901  499710.52849785] arepo_length small_a / small_h
mass per tile:  0.0020686446223407984
0.0020686446223407984 arepo_mass / small_h [499823.69781908 499886.73305113 499711.79369784] arepo_length small_a / small_h
mass per tile:  0.0028156298212707043
0.0028156298212707043 arepo_mass / small_h [499823.31962591 499884.49095515 499711.11522112] arepo_length small_a / small_h
mass per tile:  0.0038870656862854958
0.0038870656862854958 arepo_mass / small_h [499828.50279797 499885.36036159 499709.93779495] arepo_length small_a / small_h
mass per tile:  0.0038667127

In [57]:
for nx, ny, nz in pe.tile.tile_index:
    # pe.tile.particles_per_tile[nx, ny, nz]
    start_idx = pe.tile.start_index_for_tile[nx, ny, nz]
    print('mass per tile: ', pe.tile.mass_per_tile[nx, ny, nz])
    ip_orig = int(pe.tile.sort_index[start_idx])
    # print('%s'%(pe.index[ip_orig]), snap['0_Masses'][ip_orig], snap['0_Coordinates'][ip_orig])
    print(snap['0_Masses'][pe.index][ip_orig], snap['0_Coordinates'][pe.index][ip_orig])

mass per tile:  0.0035242608282715082
0.0035242608282715082 arepo_mass / small_h [499821.87123277 499882.46133265 499702.60582814] arepo_length small_a / small_h
mass per tile:  0.003444009693339467
0.003444009693339467 arepo_mass / small_h [499825.79323945 499882.43380369 499702.60233001] arepo_length small_a / small_h
mass per tile:  0.0031851318199187517
0.0031851318199187517 arepo_mass / small_h [499828.58410924 499882.54931535 499703.03114774] arepo_length small_a / small_h
mass per tile:  0.002433166839182377
0.002433166839182377 arepo_mass / small_h [499823.50170119 499886.61663805 499703.01053361] arepo_length small_a / small_h
mass per tile:  0.003085425356402993
0.003085425356402993 arepo_mass / small_h [499821.47622146 499885.84206971 499704.00528695] arepo_length small_a / small_h
mass per tile:  0.0026669474318623543
0.0026669474318623543 arepo_mass / small_h [499825.34985803 499888.39454967 499704.24873189] arepo_length small_a / small_h
mass per tile:  0.0031702360138297

In [58]:
ip_orig, start_idx

(27, array(61))

In [61]:
pe.tile.unsort_index[27]

array(61)

In [62]:
snap['0_Coordinates'][pe.index][27]

<PaicosQuantity [499825.3867262 , 499881.41019997, 499721.92274602] arepo_length small_a / small_h>

In [73]:
ip = 27
pos = snap['0_Coordinates'][pe.index].value - pe.center.value + 0.5*widths
tile_widths = pe.tile.tile_widths
xp = pos[ip, 0]
yp = pos[ip, 1]
zp = pos[ip, 2]
ip_tile_x = int(xp / tile_widths[0])
ip_tile_y = int(yp / tile_widths[1])
ip_tile_z = int(zp / tile_widths[2])
print(ip_tile_x, ip_tile_y, ip_tile_z)
# pe.index[0]

15 0 63


In [74]:
nx, ny, nz

(array(15), array(0), array(63))

In [63]:
nx, ny, nz = 10, 8, 2
start_idx = int(pe.tile.start_index_for_tile[nx, ny, nz])
print('mass per tile: ', pe.tile.mass_per_tile[nx, ny, nz])
num_part_per_tile = int(pe.tile.particles_per_tile[nx, ny, nz])

max_hsml = 0.0*arepo_length
for idx in range(start_idx, start_idx+num_part_per_tile):
    ip_orig = int(pe.tile.sort_index[idx])
    # print('%s'%(pe.index[ip_orig]), snap['0_Masses'][ip_orig], snap['0_Coordinates'][ip_orig])
    hsml = smoothing_length[pe.index][ip_orig]
    print(hsml)
    if (hsml > max_hsml):
        max_hsml = hsml
print('max_hsml: ', max_hsml)
print(pe.tile.max_hsml_per_tile[nx, ny, nz])

array([0.3125, 0.32  , 0.3125])

In [75]:
pe.tile.sort_index

array([59, 61, 60, 40, 37, 20, 58, 33, 57, 32, 12, 15, 14, 21, 25,  7, 10,
       13, 18,  5,  9,  2,  6, 43,  8,  4,  3, 16,  0, 30, 42,  1, 29, 44,
       19, 22, 11, 48, 17, 23, 26, 31, 51, 46, 39, 38, 55, 45, 41, 24, 50,
       53, 49, 47, 36, 54, 28, 56, 52, 34, 35, 27])

In [ ]:
angles = np.arange(7)*5

for angle in angles:
    U = pe.compute_potential(smoothing_length, angle=angle)
    print(f'theta: {angle:.1f} \t', 'U: ', U)

In [ ]:
np.sum(pe.index)

In [ ]:
snap_select = snap.select(pe.index, parttype=0)

In [ ]:
snap_select['0_Masses'].shape[0]

In [ ]:
def direct_calculation(snap, index, smoothing_length):
    snap_select = snap.select(pe.index, parttype=0)
    Np = snap_select['0_Masses'].shape[0]

    if isinstance(smoothing_length.value, np.ndarray):
        smoothing_length = smoothing_length[index]
    else:
        smoothing_length = np.ones(Np) * smoothing_length

    potential = 0.0

    for i in range(Np):
        if (i%1000==0):
            print(f'particle {i}')
        h = smoothing_length[i].value
        for j in range(Np):
            m1m2 = (snap_select['0_Masses'][i]*snap_select['0_Masses'][j]).value
            if (i==j):
                potential -= m1m2*(3./2.)/h
            # else:
            #     r = np.sqrt(np.sum((snap_select['0_Coordinates'][i]-snap_select['0_Coordinates'][j])**2)).value
            #     potential_tmp = 1./r
            #     if (r <= h):
            #         potential_tmp *= (0.5*r/h)*(3-(r/h)**2)
            #     potential -= potential_tmp
            

    G = 6.67430e-8 * snap_select.uq('cm^3 g^-1 s^-2')
    potential = 0.5 * G * potential * snap_select.mass**2 / snap_select.length
    return potential
    

In [ ]:
U_direct = direct_calculation(snap, pe.index, smoothing_length)

In [ ]:
U_direct

In [ ]:
G = 6.67430e-8 * snap_select.uq('cm^3 g^-1 s^-2')
print(0.5*G*np.sum(snap_select['0_Masses']**2*(3./2.)/smoothing_length))

In [ ]:
0.5*G*cp.asnumpy(np.sum(pe.tile.mass_per_tile**2))*(3./2.)/smoothing_length